# DeepLabV3+ V5 — Extraction automatique de la ligne de côte de falaises

**Auteure :** Rahima ZOUHHAD  
**Encadrants :** Michele LINARDI, Walid RABEHI  
**Contexte :** Projet de recherche M2 SIC — CY Université / ENSEA — 2026

---

## Objectif

Ce notebook implémente un pipeline de **segmentation sémantique binaire** basé sur l'architecture
**DeepLabV3+** (Chen et al., 2018) pour la détection automatique de la ligne de crête des falaises
calcaires de Normandie à partir d'images satellites Pléiades à 50 cm de résolution.

Chaque pixel de l'image est classifié comme :
- **1** : appartient à la ligne de côte (pixels de crête de falaise)
- **0** : fond (mer, falaise, plateau agricole, zone urbaine)

La tâche est rendue difficile par un **déséquilibre de classes extrême** : les pixels de côte
représentent environ 0.66% du dataset filtré.

---

## Historique des versions et corrections V5

Cette version V5 corrige trois problèmes identifiés dans V4 qui produisaient un masque d'inférence
entièrement noir (probabilités inférieures à 0.05 sur toute l'image) :

### Correction 1 — `pos_weight` automatique (~147) couplé à Focal Loss
**Problème V4 :** Le ratio brut négatifs/positifs (~150) était passé directement comme `pos_weight`
à la Focal Loss. Or la Focal Loss intègre déjà un mécanisme implicite de re-pondération via
`(1 - p_t)^gamma` qui concentre le gradient sur les exemples difficiles. Ajouter `pos_weight=147`
en plus créait une **double sur-pénalisation** des pixels côte, forçant le modèle à produire
des logits très négatifs proches de zéro → `sigmoid(logits) ≈ 0.02–0.15` → masque noir.

**Correction V5 :** `pos_weight = 10.0` fixe (même valeur que le UNet V12, empiriquement optimale).
La Tversky Loss avec `beta=0.7` compense déjà largement le déséquilibre.

### Correction 2 — `_apply_dilation()` incomplète sur ResNet-50
**Problème V4 :** La fonction ne modifiait que `conv2` et `downsample[0].stride` de chaque bloc
Bottleneck. Dans ResNet-50, les blocs d'entrée de `layer3` et `layer4` ont des strides résiduels
sur d'autres convolutions, laissant des réductions de résolution non annulées avant l'ASPP.

**Correction V5 :** `_freeze_stride_to_one()` parcourt tous les modules via `named_modules()` et
annule tout stride > 1, puis applique la dilation uniquement sur les noyaux 3×3.

### Correction 3 — `ReduceLROnPlateau` trop agressif
**Problème V4 :** `patience=7` avec validation tous les 5 epochs = possibilité de réduction du lr
après seulement 35 epochs sans amélioration, trop tôt pour un backbone en fine-tuning.

**Correction V5 :** Remplacement par `CosineAnnealingLR` (même stratégie que UNet V12),
sans risque de blocage prématuré.

## 1 — Vérification GPU

In [ ]:
# Vérifie si un GPU CUDA est disponible sur la machine.
# L'entraînement de DeepLabV3+ avec backbone ResNet-50 est très lent sur CPU
# (~plusieurs heures par epoch). Un GPU est fortement recommandé.
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'PAS DE GPU CUDA')

## 2 — Imports et chemins

**Dépendances principales :**
- `torch` / `torchvision` : framework deep learning et backbone ResNet-50 pré-entraîné
- `albumentations` : augmentation de données avec synchronisation image/masque garantie
- `rasterio` : lecture/écriture de fichiers GeoTIFF géoréférencés (images Pléiades)
- `sklearn` : calcul des courbes ROC et Precision-Recall
- `scipy.ndimage` : post-traitement morphologique (filtrage par composantes connexes, dilatation)
- `IPSKFold` : stratification IPS pour la construction des splits train/val/test
  (Jami et al., 2025 — garantit un équilibre de distribution de classes entre les splits)

In [ ]:
import os, random
import numpy as np
import torch
from torchvision import models
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm import tqdm
from glob import glob
from PIL import Image
import rasterio
from rasterio.windows import Window
from rasterio.merge import merge
from rasterio.transform import Affine
from rasterio.crs import CRS
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from scipy import ndimage
from scipy.ndimage import binary_dilation
import sys

# Librairie de stratification sémantique IPS.
# À télécharger depuis : https://github.com/SEA-AI/SemanticStratification
# L'IPS (Iterative Pixel Stratification) assure que la distribution de pixels
# par classe est équilibrée entre les splits train/val/test, ce qui est critique
# quand les pixels positifs (côte) représentent < 1% du total.
#sys.path.append(r'C:\Users\Hima\Desktop\DeeplabV3+\SemanticStratification')
sys.path.append(r'\SemanticStratification')
from stratifiers.ips import IPSKFold

# Fixation des graines aléatoires pour la reproductibilité des résultats.
# La même graine est utilisée pour Python, NumPy et PyTorch.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── À ADAPTER selon votre arborescence ──────────────────────────────────────
# image_dir : dossier contenant les tuiles image .tif (256×256 px, 8-bit RGB)
# mask_dir  : dossier contenant les tuiles masque .tif correspondantes
#             (même nom de fichier, valeurs binaires 0/1)

#os.chdir(r'C:\Users\Hima\Desktop')#A ADAPTER
data_root = r'\ ' #r'C:\Users\Hima\Desktop\DeeplabV3+'
image_dir = os.path.join(data_root, 'Images')
mask_dir  = os.path.join(data_root, 'Masks')
# ─────────────────────────────────────────────────────────────────────────────

print('Chemins OK')

## 3 — Dataset et augmentation

### Normalisation ImageNet
Le backbone ResNet-50 a été pré-entraîné sur ImageNet avec des statistiques de normalisation
spécifiques. Il est impératif d'appliquer la **même normalisation** en entrée du modèle,
faute de quoi les feature maps du backbone sont hors de leur distribution d'apprentissage.

### Transformations d'augmentation (entraînement uniquement)
L'augmentation est appliquée **stochastiquement à chaque epoch** (pas en pré-traitement fixe)
pour que le modèle voit différentes réalisations du même tile selon les epochs.

La librairie `albumentations` est utilisée car elle garantit que les transformations géométriques
sont appliquées **identiquement et simultanément** à l'image ET à son masque — ce que
`torchvision.transforms` ne garantit pas.

| Transformation | Paramètres | Justification |
|---|---|---|
| HorizontalFlip | p=0.5 | Symétrie gauche-droite sans signification géomorphologique |
| Rotate | ±10°, p=0.4 | Plus conservateur que ±15° (testé sur UNet) ; le backbone pré-entraîné est plus robuste aux rotations (Kornblith et al., 2019) |
| ColorJitter | brightness/contrast=0.2, p=0.7 | Simule la variabilité radiométrique inter-acquisitions (heure, saison, atmosphère) |

Les transformations géométriques plus agressives (random crop, bruit gaussien) ont été testées
et exclues : avec 323 tuiles d'entraînement, elles dégradent les performances
(Réf : Yu et al., 2017 — bénéfice de l'augmentation dépendant de la taille du dataset).

In [ ]:
# Statistiques ImageNet standard — obligatoires pour tout backbone pré-entraîné sur ImageNet.
# Ces valeurs correspondent à la moyenne et l'écart-type par canal (R, G, B)
# calculés sur l'ensemble d'entraînement d'ImageNet.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
TILE_SIZE     = 256  # Taille des tuiles en pixels (carré)


def get_train_transform():
    """Pipeline d'augmentation pour l'entraînement.
    
    Appliqué stochastiquement à chaque epoch. albumentations synchronise
    automatiquement les transformations géométriques entre l'image et le masque.
    La normalisation ImageNet est toujours appliquée en dernier.
    """
    return A.Compose([
        # Retournement horizontal : physiquement justifié (pas de sens géographique
        # à la direction gauche-droite d'une ligne de côte).
        A.HorizontalFlip(p=0.5),
        # Rotation légère : simule des variations de géométrie d'acquisition.
        # border_mode=0 : remplissage par des zéros (noir) aux bords
        # pour éviter les artéfacts de répétition.
        A.Rotate(limit=10, border_mode=0, p=0.4),
        # Jitter colorimétrique : simule la variabilité radiométrique
        # entre acquisitions (conditions atmosphériques, heure solaire, saison).
        # hue très faible (0.02) pour ne pas dénaturer les couleurs de l'image.
        A.ColorJitter(brightness=0.2, contrast=0.2,
                      saturation=0.05, hue=0.02, p=0.7),
        # Normalisation ImageNet : TOUJOURS en dernier, après les augmentations.
        # Convertit les valeurs uint8 [0-255] vers float32 normalisé.
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        # Conversion du numpy array HWC vers un tenseur PyTorch CHW.
        ToTensorV2(),
    ])


def get_val_transform():
    """Pipeline de transformation pour la validation, le test et l'inférence.
    
    Pas d'augmentation : uniquement la normalisation ImageNet obligatoire.
    Utilisé aussi lors du filtrage du dataset (pour cohérence des valeurs).
    """
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


class CoastlineDataset(Dataset):
    """Dataset de tuiles Pléiades 256×256 pour la détection de ligne de côte.
    
    Lit les paires (image, masque) depuis des fichiers GeoTIFF.
    - Images : 3 bandes RGB, 8-bit, valeurs [0, 255]
    - Masques : 1 bande binaire, valeurs {0, 1} (0=fond, 1=côte)
    
    Le tri alphabétique des chemins garantit la correspondance image↔masque
    (même convention de nommage requise).
    
    Note : seules les 3 premières bandes de l'image sont lues (RGB).
    La 4e bande (NIR, si présente) est ignorée — limitation de la conversion
    8-bit effectuée lors du découpage QGIS.
    """
    def __init__(self, image_dir, mask_dir, img_transform=None):
        self.image_paths = sorted(glob(os.path.join(image_dir, '*.tif')))
        self.mask_paths  = sorted(glob(os.path.join(mask_dir,  '*.tif')))
        self.transform   = img_transform
        # Vérification de cohérence : le nombre d'images doit égaler le nombre de masques.
        assert len(self.image_paths) == len(self.mask_paths), \
            f'Mismatch images/masques : {len(self.image_paths)} vs {len(self.mask_paths)}'

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Lecture de l'image via rasterio (format GeoTIFF).
        # src.read() retourne un tableau CHW (Channels, Height, Width).
        # On ne conserve que les 3 premières bandes (RGB).
        with rasterio.open(self.image_paths[idx]) as src:
            img = src.read()[:3]
        # Lecture du masque binaire (1 seule bande).
        # Conversion en float32 requis par PyTorch pour le calcul des losses.
        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1).astype(np.float32)

        # Conversion CHW → HWC (Height, Width, Channels) pour albumentations
        # qui attend des images au format HWC standard numpy/OpenCV.
        img_hwc = np.moveaxis(img.astype(np.uint8), 0, -1)
        if self.transform:
            # albumentations retourne un dict ; on récupère image et masque transformés.
            out      = self.transform(image=img_hwc, mask=mask)
            img_hwc  = out['image']   # tenseur PyTorch CHW après ToTensorV2
            mask     = out['mask']    # tenseur PyTorch HW
        return img_hwc, mask

print('Dataset et transforms établis')

## 4 — Filtrage des tuiles + Stratification IPS + DataLoaders

### Filtrage par nombre de pixels côte
Le dataset brut de 2310 tuiles (stride=64) contient une majorité de tuiles sans ligne de côte.
Ces tuiles n'apportent aucune information d'apprentissage pour la classe positive et biaiseraient
le modèle vers la prédiction systématique du fond.
On ne conserve que les tuiles contenant au moins `MIN_COAST_PIXELS = 50` pixels de côte,
ce qui réduit le dataset à **405 tuiles** (17.5%).

### Stratification IPS
La division train/val/test est réalisée via **IPSKFold** (Iterative Pixel Stratification,
Jami et al., 2025) plutôt qu'un split aléatoire classique. L'IPS itère sur les tuiles et
les assigne aux splits de façon à équilibrer la **distribution de pixels par classe** dans
chaque subset. Cela est critique ici car la densité de pixels côte varie fortement d'une
tuile à l'autre (de 50 px minimum à plusieurs centaines).

Résultat : **80% train** (323 tuiles) / **10% val** (42) / **10% test** (40),
avec ~0.66% de pixels côte dans chaque split.

### WeightedRandomSampler
En complément de l'IPS, un `WeightedRandomSampler` sur-représente les tuiles positives
dans chaque batch d'entraînement (70% de tuiles avec côte, 30% sans).
Cette stratégie d'oversampling améliore la convergence en début d'entraînement
(Réf : Buda et al., 2018).

In [ ]:
# Seuil minimal de pixels côte pour conserver une tuile.
# Valeur de 50 px : compromis entre richesse du dataset et qualité des tuiles retenues.
MIN_COAST_PIXELS = 50

# Proportion cible de tuiles positives (contenant de la côte) dans chaque batch.
# 0.7 = 70% de tuiles avec côte par batch, malgré leur minorité dans le dataset.
# Réf : Buda et al. (2018) — l'oversampling est plus efficace que la pondération
# de loss seule pour les classes minoritaires, surtout en début d'entraînement.
POS_PATCH_RATIO  = 0.7


def filter_dataset(dataset, min_coast_pixels):
    """Filtre les tuiles contenant moins de min_coast_pixels pixels de côte.
    
    Retourne la liste des indices des tuiles conservées dans le dataset original.
    """
    print(f'Filtrage (seuil = {min_coast_pixels} px)')
    kept, empty, total_coast = [], 0, 0
    for i in range(len(dataset)):
        _, mask = dataset[i]
        n_coast = mask.sum().item()
        if n_coast >= min_coast_pixels:
            kept.append(i)
            total_coast += n_coast
        else:
            empty += 1
    print(f'  Tuiles conservées : {len(kept):4d} / {len(dataset)} '
          f'({100*len(kept)/len(dataset):.1f}%)')
    print(f'  Tuiles vides exclues : {empty}')
    print(f'  Pixels côte totaux : {total_coast:,}')
    return kept


class StratificationWrapper:
    """Adaptateur pour rendre le dataset compatible avec l'interface IPSKFold.
    
    IPSKFold attend un itérable qui yield (image, masque_numpy_int64).
    L'image n'est pas utilisée par l'IPS (seul le masque compte pour
    calculer la distribution de classes), donc on yield None.
    """
    def __init__(self, indices, base_dataset):
        self.indices     = indices    # Indices des tuiles filtrées
        self.base        = base_dataset
        self.num_classes = 2          # Binaire : fond (0) et côte (1)

    def __len__(self):
        return len(self.indices)

    def __iter__(self):
        for i in self.indices:
            _, mask = self.base[i]
            # IPSKFold requiert des masques en int64 (labels discrets)
            yield None, mask.numpy().astype(np.int64)


# ── Construction du dataset de base (sans augmentation) pour le filtrage ──
# On utilise get_val_transform() (normalisation uniquement) pour que les valeurs
# de pixels soient cohérentes lors du comptage des pixels côte.
base_ds  = CoastlineDataset(image_dir, mask_dir, img_transform=get_val_transform())
kept_idx = filter_dataset(base_ds, MIN_COAST_PIXELS)

# ── Stratification IPS sur les tuiles filtrées ────────────────────────────
# n_splits=10 → fold 0 = test, fold 1 = validation, folds 2-9 = train
# shuffle=False : ordre déterministe pour reproductibilité
strat_ds  = StratificationWrapper(kept_idx, base_ds)
ips       = IPSKFold(n_splits=10, shuffle=False)
all_folds = list(ips.split(strat_ds))

# Convention : fold 0 = test, fold 1 = val, folds 2-9 = train
# Les indices retournés par IPS sont des indices LOCAUX dans kept_idx ;
# on les reconvertit en indices GLOBAUX dans le dataset original.
_, test_local  = all_folds[0]
_, val_local   = all_folds[1]
train_local    = np.concatenate([all_folds[i][1] for i in range(2, 10)])

train_idx = [kept_idx[i] for i in train_local]
val_idx   = [kept_idx[i] for i in val_local]
test_idx  = [kept_idx[i] for i in test_local]

# ── Création des datasets avec les transformations appropriées ────────────
# Train : augmentation activée | Val/Test : normalisation seule
train_dataset = Subset(
    CoastlineDataset(image_dir, mask_dir, get_train_transform()), train_idx)
val_dataset   = Subset(
    CoastlineDataset(image_dir, mask_dir, get_val_transform()), val_idx)
test_dataset  = Subset(
    CoastlineDataset(image_dir, mask_dir, get_val_transform()), test_idx)

# ── WeightedRandomSampler pour sur-représenter les tuiles positives ───────
# On calcule un poids par tuile : w_pos pour les tuiles avec côte,
# w_neg pour les tuiles sans (ou avec moins de MIN_COAST_PIXELS px de côte).
# Les poids sont inversement proportionnels à la fréquence de chaque classe
# de tuile, modulés par POS_PATCH_RATIO.
base_ds_train    = CoastlineDataset(image_dir, mask_dir, get_val_transform())
n_coast_per_tile = []
for i in train_idx:
    _, mask = base_ds_train[i]
    n_coast_per_tile.append(mask.sum().item())

is_positive = [n >= MIN_COAST_PIXELS for n in n_coast_per_tile]
# Poids inversement proportionnels à la fréquence, bornés par POS_PATCH_RATIO
w_pos = POS_PATCH_RATIO / (sum(is_positive) + 1e-8)
w_neg = (1 - POS_PATCH_RATIO) / (sum(not p for p in is_positive) + 1e-8)
weights = [w_pos if p else w_neg for p in is_positive]
# replacement=True : tirage avec remise (oversampling possible)
sampler = WeightedRandomSampler(
    weights=torch.tensor(weights, dtype=torch.float32),
    num_samples=len(weights),
    replacement=True)

# ── DataLoaders ───────────────────────────────────────────────────────────
# Train : sampler personnalisé (pas shuffle — incompatible avec WeightedRandomSampler)
# Val/Test : pas de sampler (évaluation déterministe)
# num_workers=0 : chargement en séquentiel (compatible Windows/CPU)
train_dl = DataLoader(train_dataset, batch_size=8, sampler=sampler,
                      num_workers=0, pin_memory=False)
val_dl   = DataLoader(val_dataset,   batch_size=8, shuffle=False,
                      num_workers=0, pin_memory=False)
test_dl  = DataLoader(test_dataset,  batch_size=8, shuffle=False,
                      num_workers=0, pin_memory=False)

# Affichage du résumé des splits pour vérification de l'équilibre IPS
for name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    n_px = sum(base_ds[i][1].sum().item() for i in idx)
    pct  = n_px / (256 * 256 * len(idx)) * 100
    print(f'  {name:10}: {len(idx):3d} tuiles | {n_px:,.0f} px côte ({pct:.2f}%)')

## 5 — Architecture DeepLabV3+

### Principe général
DeepLabV3+ (Chen et al., ECCV 2018) combine :
1. Un **backbone encodeur** (ResNet-50) qui extrait des features multi-échelles
2. Un module **ASPP** (Atrous Spatial Pyramid Pooling) qui capture le contexte
   multi-échelle sans réduire la résolution spatiale
3. Un **décodeur léger** qui affine les frontières en fusionnant les features
   ASPP avec les features bas-niveau du backbone

Contrairement au U-Net qui utilise des skip connections à chaque niveau de l'encodeur,
DeepLabV3+ n'utilise qu'**une seule connexion** (depuis `layer1`, résolution 1/4)
et s'appuie sur l'ASPP pour le contexte multi-échelle.

### Taux de dilatation ASPP : (3, 6, 12) au lieu de (6, 12, 18)
Les taux originaux (6, 12, 18) sont calibrés pour des entrées ~512 px (PASCAL VOC, Cityscapes).
Sur des tuiles 256×256, des taux plus petits capturent mieux les structures locales
à l'échelle de notre image satellite.
Réf : Chen et al. (2018), Section 4.

### Correction V5 : `_freeze_stride_to_one()` remplace `_apply_dilation()`
Voir docstring de la méthode ci-dessous.

In [ ]:
# Sélection automatique du device : GPU si disponible, CPU sinon.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')


class ASPPConv(nn.Sequential):
    """Branche convolutionnelle dilatée de l'ASPP.
    
    Une convolution 3×3 avec un taux de dilatation r capture le contexte
    dans un champ récepteur effectif de (2r+1)×(2r+1) pixels sans
    réduire la résolution spatiale.
    padding=dilation assure que la taille de sortie est identique à l'entrée.
    bias=False : BatchNorm (qui suit) a son propre terme de biais.
    """
    def __init__(self, in_ch, out_ch, dilation):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, 3, padding=dilation,
                      dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))


class ASPPPooling(nn.Sequential):
    """Branche de pooling global de l'ASPP.
    
    AdaptiveAvgPool2d(1) réduit la feature map à 1×1 (contexte global),
    puis une Conv2d 1×1 projette les canaux. L'interpolation bilinéaire
    restaure la résolution spatiale d'origine.
    Cette branche capture le contexte sémantique global de la tuile.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        size = x.shape[-2:]  # Taille spatiale à restaurer après pooling global
        for mod in self:
            x = mod(x)
        # Interpolation bilinéaire pour revenir à la taille originale
        return F.interpolate(x, size=size, mode='bilinear', align_corners=False)


class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling — module central de DeepLabV3+.
    
    Capture le contexte multi-échelle en appliquant en parallèle :
    - 1 conv 1×1 (contexte ponctuel)
    - 3 conv 3×3 dilatées (taux r=3, 6, 12 pour tuiles 256px)
    - 1 pooling global (contexte de la tuile entière)
    
    Les 5 sorties (chacune de 256 canaux) sont concaténées → 1280 canaux,
    puis projetées vers 256 canaux par une conv 1×1 + Dropout(0.5).
    
    Taux (3,6,12) au lieu de (6,12,18) : adapté aux tuiles 256×256 px.
    Réf : Chen et al. (2018), DeepLabV3+, ECCV, Section 4.
    """
    def __init__(self, in_ch, out_ch=256, rates=(3, 6, 12)):
        super().__init__()
        # Conv 1×1 : capture le contexte ponctuel (sans dilatation)
        modules = [nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))]
        # Convolutions dilatées : un module ASPPConv par taux
        for r in rates:
            modules.append(ASPPConv(in_ch, out_ch, r))
        # Branche de pooling global
        modules.append(ASPPPooling(in_ch, out_ch))
        self.convs   = nn.ModuleList(modules)
        # Projection de la concaténation : (len(rates)+2)*256 → 256 canaux
        # Dropout(0.5) : régularisation forte sur la représentation ASPP
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * (len(rates) + 2), out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True), nn.Dropout(0.5))

    def forward(self, x):
        # Application parallèle de toutes les branches, puis concaténation
        return self.project(torch.cat([c(x) for c in self.convs], dim=1))


class DeepLabV3Plus(nn.Module):
    """DeepLabV3+ pour segmentation binaire avec backbone ResNet-50 pré-entraîné.
    
    Architecture :
    - Encodeur : ResNet-50 (couches layer0 à layer4)
    - ASPP : capture le contexte multi-échelle depuis layer4 (2048 canaux)
    - Projection bas-niveau : réduit layer1 de 256 à 48 canaux
    - Décodeur : fusionne ASPP (upsamplé) + features bas-niveau (48 canaux)
                 → 304 canaux → 2 convolutions 3×3 → 1 logit de sortie
    - Sortie : logits bruts (pas de sigmoid — sigmoid appliqué dans la loss
               et lors de l'inférence pour éviter des instabilités numériques)
    
    Modifications V5 :
    - _apply_dilation() remplacé par _freeze_stride_to_one() (voir docstring)
    - Taux ASPP : (3,6,12) inchangé par rapport à V4
    """
    def __init__(self, backbone='resnet50', pretrained=True, num_classes=1):
        super().__init__()
        # Chargement du backbone ResNet-50 pré-entraîné sur ImageNet
        resnet = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        
        # Décomposition du ResNet-50 en couches séparées pour accès aux
        # features intermédiaires (nécessaire pour la connexion bas-niveau).
        # layer0 : conv1 + BN + ReLU + MaxPool (stride total = 4 → 256px → 64px)
        self.layer0 = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        # layer1 : 3 blocs Bottleneck, 256 canaux de sortie, résolution 64×64
        # → utilisé pour la connexion bas-niveau du décodeur
        self.layer1 = resnet.layer1
        # layer2 : 4 blocs Bottleneck, 512 canaux, résolution 32×32
        self.layer2 = resnet.layer2
        # layer3 : 6 blocs Bottleneck, 1024 canaux
        # → dilation=2 appliquée (stride annulé) : conserve la résolution 64×64
        self.layer3 = resnet.layer3
        # layer4 : 3 blocs Bottleneck, 2048 canaux
        # → dilation=4 appliquée (stride annulé) : conserve la résolution 64×64
        self.layer4 = resnet.layer4

        # Application de la dilatation sur layer3 (dilation=2) et layer4 (dilation=4).
        # Cela transforme le ResNet-50 classifiant (stride total 32) en réseau
        # pleinement convolutionnel avec stride effectif 8 (résolution 32× supérieure).
        self._freeze_stride_to_one(self.layer3, dilation=2)
        self._freeze_stride_to_one(self.layer4, dilation=4)

        # Module ASPP : entrée 2048 canaux (sortie layer4), sortie 256 canaux
        self.aspp     = ASPP(2048, 256, rates=(3, 6, 12))
        
        # Projection des features bas-niveau (layer1) : 256 → 48 canaux
        # Réduction intentionnelle : les features bas-niveau apportent
        # des détails spatiaux, pas du contexte sémantique.
        # Ratio 256:48 ≈ 5:1 : équilibre entre détails et contexte dans le décodeur.
        self.low_proj = nn.Sequential(
            nn.Conv2d(256, 48, 1, bias=False),
            nn.BatchNorm2d(48), nn.ReLU(inplace=True))
        
        # Décodeur : fusionne ASPP upsamplé (256 ch) + features bas-niveau (48 ch)
        # = 304 canaux en entrée → 256 → 256 → 1 logit
        # Dropout(0.5) après la 1ère conv : régularisation forte
        # Dropout(0.1) après la 2ème conv : régularisation légère avant la sortie
        self.decoder  = nn.Sequential(
            nn.Conv2d(304, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.Dropout(0.1),
            nn.Conv2d(256, num_classes, 1))  # Projection finale vers 1 logit

    def _freeze_stride_to_one(self, layer, dilation):
        """Convertit une couche ResNet de classification en couche dilatée.
        
        VERSION V5 — Remplace _apply_dilation() de V4.
        
        Problème de V4 : _apply_dilation() ne modifiait que conv2 et
        downsample[0].stride de chaque bloc Bottleneck. Dans ResNet-50,
        certains blocs d'entrée de layer3/layer4 ont des strides résiduels
        sur d'autres convolutions (notamment le downsample), laissant des
        réductions de résolution non annulées → feature maps de taille
        incorrecte transmises à l'ASPP.
        
        Correction V5 : parcourt TOUS les modules via named_modules() et :
          1. Force stride=(1,1) sur toute Conv2d dont le stride est >1,
             annulant toutes les réductions de résolution résiduelles.
          2. Applique la dilation sur les convolutions 3×3 uniquement
             (kernel_size==(3,3)), avec le padding correspondant.
             Les conv 1×1 n'ont pas de padding spatial et ne bénéficient
             pas de la dilatation, conformément à Chen et al. (2018).
        
        Résultat attendu (vérifiable par le print de shapes ci-dessous) :
        layer3 et layer4 conservent la résolution spatiale 64×64
        au lieu de la réduire à 32×32 et 16×16.
        
        Réf : Chen et al. (2018), DeepLabV3+, ECCV 2018, Section 4.
        """
        for name, module in layer.named_modules():
            if isinstance(module, nn.Conv2d):
                # Étape 1 : annule tout stride résiduel
                if module.stride != (1, 1):
                    module.stride = (1, 1)
                # Étape 2 : applique la dilation sur les noyaux 3×3 uniquement
                if module.kernel_size == (3, 3):
                    module.dilation = (dilation, dilation)
                    module.padding  = (dilation, dilation)

    def forward(self, x):
        """Passe avant du réseau.
        
        Flux :
        1. Encodeur : x → layer0 → layer1(low) → layer2 → layer3 → layer4
        2. ASPP sur la sortie de layer4 (2048 canaux, 64×64)
        3. Upsampling ASPP vers la résolution de layer1 (64×64 → même taille)
        4. Projection des features bas-niveau layer1 (256 → 48 canaux)
        5. Concaténation ASPP + bas-niveau → décodeur → logit unique
        6. Upsampling final vers la résolution d'entrée (256×256)
        """
        size = x.shape[-2:]       # Taille d'entrée : (256, 256) — à restaurer en sortie
        x    = self.layer0(x)     # → (batch, 64, 64, 64) avec stride=4
        low  = self.layer1(x)     # → (batch, 256, 64, 64) — features bas-niveau
        x    = self.layer2(low)   # → (batch, 512, 32, 32)
        x    = self.layer3(x)     # → (batch, 1024, 64, 64) avec dilation=2 (V5)
        x    = self.layer4(x)     # → (batch, 2048, 64, 64) avec dilation=4 (V5)
        x    = self.aspp(x)       # → (batch, 256, 64, 64)
        # Upsampling ASPP vers la résolution des features bas-niveau
        x    = F.interpolate(x, size=low.shape[-2:],
                              mode='bilinear', align_corners=False)
        low  = self.low_proj(low) # → (batch, 48, 64, 64)
        # Concaténation : (256 + 48) = 304 canaux
        x    = self.decoder(torch.cat([x, low], dim=1))  # → (batch, 1, 64, 64)
        # Upsampling final vers la taille d'entrée (256×256)
        # Retourne des LOGITS bruts (avant sigmoid) pour compatibilité avec
        # binary_cross_entropy_with_logits (plus stable numériquement).
        return F.interpolate(x, size=size,
                             mode='bilinear', align_corners=False)


model = DeepLabV3Plus(backbone='resnet50', pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres entraînables : {n_params:,}')

# ── Vérification des résolutions intermédiaires (sanity check V5) ─────────
# Après correction, layer3 et layer4 doivent conserver la résolution 64×64.
# Si on voit 32×32 ou 16×16, la correction _freeze_stride_to_one() a échoué.
with torch.no_grad():
    dummy = torch.zeros(1, 3, 256, 256).to(device)
    x0  = model.layer0(dummy)
    l1  = model.layer1(x0)
    l2  = model.layer2(l1)
    l3  = model.layer3(l2)
    l4  = model.layer4(l3)
    print(f'Résolutions backbone : layer0={x0.shape[-2:]} | '
          f'layer1={l1.shape[-2:]} | layer2={l2.shape[-2:]} | '
          f'layer3={l3.shape[-2:]} | layer4={l4.shape[-2:]}')
    # Résultat attendu V5 : layer3=64×64, layer4=64×64
    # Résultat V4 (bugué) : layer3=32×32, layer4=16×16

## 6 — Fonction de perte : Focal + Tversky

### Pourquoi une loss différente du U-Net ?
Le U-Net V12 utilise BCE+Dice avec `pos_weight=10`. Ce choix est adapté à un modèle
entraîné **from scratch** qui commence sans a priori sur les classes.

Le backbone ResNet-50 pré-entraîné sur ImageNet est déjà **très confiant** sur les pixels
de fond dès les premières epochs : ces pixels constituent des "exemples faciles" au sens
de Lin et al. (2017). La BCE standard ne pénalise pas cette sur-confiance et laisse
le modèle ignorer la classe minoritaire (côte).

### Focal Loss (Lin et al., 2017)
Introduit un facteur modulant `(1 - p_t)^gamma` qui **réduit la contribution des exemples
faciles** (fond très confiant, p_t ≈ 1 → facteur ≈ 0) et **concentre le gradient
sur les exemples difficiles** (pixels côte mal classifiés, p_t ≈ 0 → facteur ≈ 1).
Avec `gamma=2`, un exemple classifié à 0.9 de confiance contribue ~100× moins qu'un
exemple classifié à 0.1.

### Tversky Loss (Salehi et al., 2017)
Généralisation de la Dice Loss qui permet de pondérer différemment les faux positifs (FP)
et les faux négatifs (FN) :
- `alpha=0.3` : pénalise modérément les FP (sur-détections)
- `beta=0.7` : pénalise fortement les FN (lignes de côte manquées)

Ce choix asymétrique est justifié géomorphologiquement : manquer la ligne de côte
est plus grave que légèrement la sur-détecter.

### Correction V5 : `pos_weight = 10.0` fixe
En V4, `pos_weight` était calculé automatiquement comme le ratio brut (~147).
Ce ratio élevé, combiné à la Focal Loss qui corrige déjà le déséquilibre,
créait une double sur-pénalisation → logits très négatifs → sigmoid ≈ 0.02 → masque noir.
`pos_weight=10` (même valeur que le U-Net) est empiriquement optimal ici.

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss binaire pour la gestion du déséquilibre de classes.
    
    Formule : FL(p_t) = -(1 - p_t)^gamma * log(p_t)
    où p_t = sigmoid(logit) si y=1, 1-sigmoid(logit) si y=0.
    
    Le facteur (1-p_t)^gamma down-weight les exemples faciles :
    - p_t = 0.9 (fond confiant) → facteur = 0.1^2 = 0.01
    - p_t = 0.1 (côte mal classifiée) → facteur = 0.9^2 = 0.81
    
    Réf : Lin et al. (2017), Focal Loss for Dense Object Detection, ICCV.
    """
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma      = gamma       # Facteur de focalisation
        self.pos_weight = pos_weight  # Pondération additionnelle de la classe positive

    def forward(self, logits, targets):
        # Calcul de la BCE de base avec pondération optionnelle de la classe positive
        # reduction='none' : on garde les valeurs pixel par pixel pour appliquer
        # le facteur modulant (1-p_t)^gamma
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none')
        # p_t : probabilité de la classe vraie
        # = sigmoid(logit) pour les pixels positifs (targets=1)
        # = 1-sigmoid(logit) pour les pixels négatifs (targets=0)
        p_t = (torch.sigmoid(logits) * targets +
               (1 - torch.sigmoid(logits)) * (1 - targets))
        # Application du facteur modulant et moyenne sur tous les pixels
        return (((1 - p_t) ** self.gamma) * bce).mean()


class TverskyLoss(nn.Module):
    """Tversky Loss — généralisation asymétrique de la Dice Loss.
    
    Formule : TL = 1 - (TP + smooth) / (TP + alpha*FP + beta*FN + smooth)
    - alpha < beta : pénalise plus les FN (lignes manquées) que les FP (sur-détection)
    - smooth : évite la division par zéro sur les tuiles sans pixel côte
    
    Choix alpha=0.3, beta=0.7 : asymétrie forte en faveur du rappel.
    Manquer la ligne de côte est geomorphologiquement plus grave que la sur-détecter.
    
    Réf : Salehi et al. (2017), Tversky Loss Function for Image Segmentation.
    """
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha  = alpha   # Poids des faux positifs (faible → tolérant aux FP)
        self.beta   = beta    # Poids des faux négatifs (élevé → pénalise les FN)
        self.smooth = smooth  # Terme de lissage pour la stabilité numérique

    def forward(self, logits, targets):
        p    = torch.sigmoid(logits)  # Probabilités prédites ∈ [0, 1]
        # dims adaptés à la forme du tenseur : (1,2) pour [B,H,W] ou (1,2,3) pour [B,1,H,W]
        dims = tuple(range(1, p.dim()))
        tp   = (p * targets).sum(dim=dims)           # Vrais positifs
        fp   = (p * (1 - targets)).sum(dim=dims)     # Faux positifs
        fn   = ((1 - p) * targets).sum(dim=dims)     # Faux négatifs
        return (1 - (tp + self.smooth) /
                (tp + self.alpha * fp + self.beta * fn + self.smooth)).mean()


class FocalTverskyLoss(nn.Module):
    """Loss combinée : Focal (50%) + Tversky (50%).
    
    La Focal Loss assure un gradient stable sur les pixels difficiles.
    La Tversky Loss optimise directement le chevauchement prédit/GT
    avec une pénalisation asymétrique FP/FN adaptée aux structures linéaires.
    
    Correction V5 : pos_weight=10.0 fixe (au lieu du ratio automatique ~147 de V4).
    La Focal Loss réduit déjà implicitement la contribution des pixels de fond
    via (1-p_t)^gamma. pos_weight=147 créait une double sur-pénalisation
    → logits très négatifs → probas ≈ 0.02 → masque d'inférence entièrement noir.
    
    Réf : Abraham & Khan (2019), Focal Tversky Loss with Improved Attention U-Net, ISBI.
    """
    def __init__(self, gamma=2.0, alpha=0.3, beta=0.7,
                 pos_weight=None, w_focal=0.5):
        super().__init__()
        self.w_focal = w_focal  # Poids de la composante Focal (0.5 = égal)
        self.focal   = FocalLoss(gamma, pos_weight)
        self.tversky = TverskyLoss(alpha, beta)

    def forward(self, logits, targets):
        return (self.w_focal * self.focal(logits, targets) +
                (1 - self.w_focal) * self.tversky(logits, targets))


# ── Instanciation de la loss ──────────────────────────────────────────────
# pos_weight=10.0 FIXE — ne pas recalculer automatiquement comme en V4.
# Justification : voir docstring FocalTverskyLoss ci-dessus.
pos_weight_value = 10.0
pw = torch.tensor([pos_weight_value]).to(device)
print(f'pos_weight fixé à : {pos_weight_value} (V4 utilisait ~147 → masque noir)')

FOCAL_GAMMA   = 2.0   # Focalisation standard (Lin et al., 2017)
TVERSKY_ALPHA = 0.3   # Faible pénalisation des FP
TVERSKY_BETA  = 0.7   # Forte pénalisation des FN

criterion = FocalTverskyLoss(
    gamma=FOCAL_GAMMA, alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA,
    pos_weight=pw)
print(f'Loss : FocalTversky | gamma={FOCAL_GAMMA} | '
      f'alpha={TVERSKY_ALPHA} | beta={TVERSKY_BETA} | pos_weight={pos_weight_value}')

## 7 — Métriques d'évaluation

Toutes les métriques sont calculées sur les **probabilités brutes** (avant binarisation
au seuil optimal) sauf indication contraire. Le seuil par défaut de 0.5 est utilisé
pour les métriques pendant l'entraînement ; le seuil optimal (dérivé de la courbe PR)
est utilisé pour l'évaluation finale.

**Note sur l'accuracy :** avec ~0.66% de pixels côte, un classifieur trivial prédit
tout comme fond et obtient accuracy > 0.993. L'accuracy n'est donc **pas** un indicateur
pertinent ici — elle est reportée uniquement pour illustrer ce biais.

In [ ]:
def compute_IoU_binary(preds, masks):
    """IoU (Intersection over Union) pour la classe côte uniquement.
    IoU = TP / (TP + FP + FN)
    Seuil de binarisation fixé à 0.5.
    """
    pb    = (preds > 0.5).float()
    inter = (pb * masks).sum()
    union = pb.sum() + masks.sum() - inter
    return (inter / (union + 1e-8)).item()


def compute_mIoU_binary(preds, masks):
    """Mean IoU sur les deux classes (côte + fond).
    Retourne (mIoU, IoU_côte, IoU_fond).
    Le mIoU est la métrique standard pour la segmentation sémantique.
    """
    pb = (preds > 0.5).float()
    tp = (pb * masks).sum()
    fp = (pb * (1 - masks)).sum()
    fn = ((1 - pb) * masks).sum()
    iou_coast = tp / (tp + fp + fn + 1e-8)
    tn = ((1 - pb) * (1 - masks)).sum()
    # IoU du fond : tn / (tn + fn + fp) — symétrique à IoU_côte avec classes inversées
    iou_bg = tn / (tn + fn + fp + 1e-8)
    return ((iou_coast + iou_bg) / 2).item(), iou_coast.item(), iou_bg.item()


def compute_precision(preds, masks):
    """Précision = TP / (TP + FP).
    Proportion de pixels prédits côte qui sont vraiment de la côte.
    """
    pb = (preds > 0.5).float()
    tp = (pb * masks).sum()
    fp = (pb * (1 - masks)).sum()
    return (tp / (tp + fp + 1e-8)).item()


def compute_recall(preds, masks):
    """Rappel = TP / (TP + FN).
    Proportion de pixels côte GT qui sont correctement détectés.
    Métrique critique ici : manquer la côte est geomorphologiquement problématique.
    """
    pb = (preds > 0.5).float()
    tp = (pb * masks).sum()
    fn = ((1 - pb) * masks).sum()
    return (tp / (tp + fn + 1e-8)).item()


def compute_f1(preds, masks):
    """F1-score = 2 * Précision * Rappel / (Précision + Rappel).
    Métrique principale de comparaison entre modèles.
    Moyenne harmonique qui pénalise les déséquilibres précision/rappel.
    """
    p, r = compute_precision(preds, masks), compute_recall(preds, masks)
    return 2 * p * r / (p + r + 1e-8)


def compute_accuracy(preds, masks):
    """Accuracy globale pixel-à-pixel.
    ATTENTION : métrique trompeuse avec déséquilibre de classes.
    Un classifieur trivial (tout fond) atteint ~0.993 sans détecter une seule côte.
    Reportée uniquement pour comparaison avec la littérature.
    """
    return ((preds > 0.5).float() == masks).float().mean().item()


def evaluate_loader(model, dataloader, device, seuil=0.5):
    """Évalue le modèle sur un DataLoader complet et affiche toutes les métriques.
    
    Paramètres
    ----------
    seuil : float
        Seuil de binarisation pour le comptage GT/Pred.
        Utiliser seuil=0.5 pendant l'entraînement, seuil=seuil_optimal
        pour l'évaluation finale.
    """
    model.eval()
    all_preds, all_masks = [], []
    with torch.no_grad():  # Désactive le calcul des gradients (inférence uniquement)
        for imgs, masks in dataloader:
            imgs, masks = imgs.to(device), masks.to(device)
            # sigmoid appliqué ici pour convertir les logits en probabilités [0,1]
            probs = torch.sigmoid(model(imgs)).squeeze(1)
            all_preds.append(probs.cpu())
            all_masks.append(masks.cpu())
    # Concaténation de tous les batches en un seul tenseur
    preds = torch.cat(all_preds)
    masks = torch.cat(all_masks)
    mIoU, iou_coast, _ = compute_mIoU_binary(preds, masks)
    n_gt   = masks.sum().item()
    n_pred = ((preds > seuil).float()).sum().item()
    print(f'  {len(dataloader.dataset)} images | GT={n_gt:,.0f}px | '
          f'Pred={n_pred:,.0f}px')
    print(f'  IoU_côte={iou_coast:.3f} | mIoU={mIoU:.3f} | '
          f'F1={compute_f1(preds,masks):.3f} | '
          f'Prec={compute_precision(preds,masks):.3f} | '
          f'Rec={compute_recall(preds,masks):.3f}')
    return {
        'IoU_côte' : iou_coast,
        'mIoU'     : mIoU,
        'f1'       : compute_f1(preds, masks),
        'precision': compute_precision(preds, masks),
        'recall'   : compute_recall(preds, masks),
        'accuracy' : compute_accuracy(preds, masks),
    }


print('Métriques prêtes.')

## 8 — Entraînement

### Learning rate différencié (discriminative fine-tuning)
Le backbone ResNet-50 a été pré-entraîné sur ImageNet et possède déjà de bonnes
représentations bas-niveau. Le fine-tuner avec un lr trop élevé détruirait ces
représentations. On utilise donc :
- **lr = 1e-5** pour le backbone (layers 0-4) : mise à jour très lente
- **lr = 1e-4** pour la tête (ASPP, low_proj, décodeur) : apprentissage normal

Réf : Howard & Ruder (2018), ULMFiT, ACL 2018.

### Scheduler : CosineAnnealingLR (Correction V5)
Remplace `ReduceLROnPlateau` de V4 qui réduisait le lr de façon prématurée.
Le cosinus fait décroître le lr de lr_max à eta_min=1e-6 sur toute la durée
de l'entraînement, de façon lisse et monotone.

### Early stopping
Basé sur le F1-score de validation (meilleure métrique pour les classes déséquilibrées).
Le meilleur checkpoint est sauvegardé automatiquement.
patience=15 évaluations (=15×5=75 epochs max sans amélioration).

In [ ]:
# Nombre max d'epochs. Mis à 40 pour ce run (80 recommandé si GPU disponible).
num_epochs = 40  # 80 epochs recommandés pour convergence complète
PATIENCE   = 15  # Nombre d'évaluations val sans amélioration avant arrêt

# Réinstanciation du modèle pour repartir de zéro à chaque run
model = DeepLabV3Plus(backbone='resnet50', pretrained=True).to(device)

# ── Optimiseur Adam avec lr différencié backbone vs tête ──────────────────
# Chaque groupe de paramètres reçoit son propre lr.
# weight_decay=1e-5 : régularisation L2 sur tous les paramètres (anti-surapprentissage).
optimizer = torch.optim.Adam([
    {'params': model.layer0.parameters(), 'lr': 1e-5},  # Backbone : fine-tuning lent
    {'params': model.layer1.parameters(), 'lr': 1e-5},
    {'params': model.layer2.parameters(), 'lr': 1e-5},
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.aspp.parameters(),    'lr': 1e-4}, # Tête : apprentissage normal
    {'params': model.low_proj.parameters(),'lr': 1e-4},
    {'params': model.decoder.parameters(), 'lr': 1e-4},
], weight_decay=1e-5)

# ── Scheduler CosineAnnealingLR (Correction V5) ───────────────────────────
# T_max=num_epochs : le lr décroit sur toute la durée de l'entraînement.
# eta_min=1e-6 : lr minimal en fin d'entraînement.
# step() appelé à chaque epoch (pas à chaque validation comme ReduceLROnPlateau).
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs, eta_min=1e-6)

# Variables de suivi de l'entraînement
best_val_f1   = -1.0   # Meilleur F1 val observé (critère de sauvegarde)
patience_ctr  = 0      # Compteur d'évaluations sans amélioration
loss_list     = []     # Loss moyenne par epoch (train)
val_loss_list = []     # Loss de validation (epoch, valeur) — pour courbe

print(f'epochs={num_epochs} | patience={PATIENCE} | device={device}')
print(f'lr backbone=1e-5 | lr tête=1e-4 | scheduler=CosineAnnealingLR')
print(f'pos_weight={pos_weight_value} | Focal gamma={FOCAL_GAMMA} | '
      f'Tversky alpha={TVERSKY_ALPHA} beta={TVERSKY_BETA}')

for epoch in range(num_epochs):
    # ── Phase d'entraînement ──────────────────────────────────────────────
    model.train()  # Active BatchNorm et Dropout en mode entraînement
    running_loss, n_batches = 0.0, 0
    pbar = tqdm(train_dl, desc=f'Epoch {epoch+1}/{num_epochs}')
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()  # Remet les gradients à zéro avant chaque batch
        # forward : model retourne des logits [batch, 1, H, W]
        # squeeze(1) : supprime la dimension des canaux → [batch, H, W]
        loss = criterion(model(imgs).squeeze(1), masks)
        loss.backward()  # Calcul des gradients par rétropropagation
        # Gradient clipping : limite la norme L2 des gradients à 1.0
        # Évite les explosions de gradient fréquentes avec les backbones pré-entraînés
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()  # Mise à jour des paramètres
        running_loss += loss.item()
        n_batches    += 1
        # Affiche le lr de la tête (groupe 5 = ASPP) pour suivi en temps réel
        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'lr'  : f'{optimizer.param_groups[5]["lr"]:.1e}'})

    avg = running_loss / n_batches
    loss_list.append(avg)
    # step() à chaque epoch pour CosineAnnealingLR (contrairement à ReduceLROnPlateau
    # qui se basait sur une métrique de validation)
    scheduler.step()

    print(f'Epoch {epoch+1:3d}/{num_epochs} | Loss={avg:.4f} | '
          f'lr_tête={optimizer.param_groups[5]["lr"]:.2e}')

    # ── Phase de validation (tous les 5 epochs et à la dernière epoch) ────
    if (epoch + 1) % 5 == 0 or (epoch + 1) == num_epochs:
        print(f'  Validation...')
        model.eval()  # Désactive BatchNorm/Dropout en mode évaluation
        val_running_loss, val_n_batches = 0.0, 0
        with torch.no_grad():
            for imgs_v, masks_v in val_dl:
                imgs_v, masks_v = imgs_v.to(device), masks_v.to(device)
                v_loss = criterion(model(imgs_v).squeeze(1), masks_v)
                val_running_loss += v_loss.item()
                val_n_batches    += 1
        avg_val_loss = val_running_loss / val_n_batches
        val_loss_list.append((epoch + 1, avg_val_loss))
        print(f'  Val Loss={avg_val_loss:.4f}')

        # Calcul du F1 sur le split de validation (critère de sauvegarde)
        val_m = evaluate_loader(model, val_dl, device)

        # Sauvegarde du meilleur modèle (checkpoint) basée sur le F1 val
        if val_m['f1'] > best_val_f1:
            best_val_f1  = val_m['f1']
            patience_ctr = 0
            torch.save(model.state_dict(), 'best_model_deeplabv3plus_v5.pth')
            print(f'  Meilleur F1 val={best_val_f1:.3f} → sauvegardé')
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'Early stopping (patience={PATIENCE})')
                break

# Sauvegarde du modèle final (dernière epoch), distinct du meilleur checkpoint
torch.save(model.state_dict(),
           f'deeplabv3plus_v5_final_epochs_{num_epochs}.pth')
print(f'Entraînement terminé. Meilleur F1 val : {best_val_f1:.3f}')

## 9 — Courbes ROC et Precision-Recall + seuil optimal de binarisation

Le seuil optimal est défini comme le seuil maximisant le F1-score sur le split de validation.
Il est utilisé pour l'évaluation finale sur les splits train/val/test.

**Attention :** Ce seuil optimal sur tuiles ne correspond pas nécessairement au seuil
opérationnel optimal en inférence sliding-window (les jonctions entre tuiles reçoivent
des probabilités moyennées, souvent plus faibles). En inférence, on utilise SEUIL=0.5.

In [ ]:
def plot_roc(model, dataloader, device,
             save_path=f'roc_curve_deeplabv3plus_v5_{num_epochs}.png'):
    """Calcule et trace les courbes ROC et Precision-Recall sur un DataLoader.
    
    Retourne le seuil optimal de binarisation (maximisant le F1 sur la validation).
    Affiche également la distribution des probabilités prédites (diagnostic).
    """
    model.eval()
    all_probs_list, all_labels_list = [], []
    with torch.no_grad():
        for imgs, masks in dataloader:
            # flatten() : aplatit les tenseurs [batch, H, W] en vecteurs 1D
            # pour sklearn qui attend des tableaux 1D
            probs  = torch.sigmoid(
                model(imgs.to(device))).cpu().numpy().flatten()
            labels = masks.numpy().flatten()
            all_probs_list.append(probs)
            all_labels_list.append(labels)

    all_probs  = np.concatenate(all_probs_list)   # Toutes les probabilités prédites
    all_labels = np.concatenate(all_labels_list)  # Labels GT correspondants

    # Calcul de la courbe Precision-Recall et du seuil optimal
    precisions, recalls, thresholds = precision_recall_curve(
        all_labels, all_probs)
    f1_scores     = 2 * precisions[:-1] * recalls[:-1] / (
        precisions[:-1] + recalls[:-1] + 1e-8)
    optimal_idx   = np.argmax(f1_scores)   # Indice du seuil maximisant F1
    optimal_seuil = thresholds[optimal_idx]

    # Calcul de la courbe ROC et de l'AUC
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc     = auc(fpr, tpr)

    # ── Tracé des deux courbes côte à côte ────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.plot(fpr, tpr, color='steelblue', lw=2,
             label=f'AUC = {roc_auc:.3f}')
    ax1.plot([0, 1], [0, 1], 'k--', lw=1)  # Ligne de référence (classifieur aléatoire)
    ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR (recall)')
    ax1.set_title('Courbe ROC — DeepLabV3+ V5')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(recalls, precisions, color='coral', lw=2)
    ax2.scatter(recalls[optimal_idx], precisions[optimal_idx],
                color='red', zorder=5,
                label=f'Seuil de binarisation = {optimal_seuil:.3f}\n'
                      f'F1={f1_scores[optimal_idx]:.3f} '
                      f'Prec={precisions[optimal_idx]:.3f} '
                      f'Rec={recalls[optimal_idx]:.3f}')
    ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
    ax2.set_title('Courbe Precision-Recall — DeepLabV3+ V5')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()

    print(f'AUC                : {roc_auc:.3f}')
    print(f'Seuil binarisation : {optimal_seuil:.3f}')
    print(f'  Precision        : {precisions[optimal_idx]:.3f}')
    print(f'  Recall           : {recalls[optimal_idx]:.3f}')
    print(f'  F1               : {f1_scores[optimal_idx]:.3f}')

    # ── Diagnostic de la distribution des probabilités ────────────────────
    # Permet de détecter le problème V4 (probas toutes < 0.1 → masque noir)
    coast_probs = all_probs[all_labels == 1]  # Probas sur les pixels côte GT
    bg_probs    = all_probs[all_labels == 0]  # Probas sur les pixels fond GT
    print(f'\nDistribution des probabilités (sur tuiles val) :')
    print(f'  Pixels côte  : max={coast_probs.max():.3f} | '
          f'mean={coast_probs.mean():.3f} | p50={np.median(coast_probs):.3f}')
    print(f'  Pixels fond  : max={bg_probs.max():.3f} | '
          f'mean={bg_probs.mean():.3f} | p99={np.percentile(bg_probs, 99):.3f}')
    print(f'  → Si max côte > 0.5 et bien séparé du fond : inférence seuil=0.5 OK')

    return optimal_seuil


# Calcul du seuil optimal sur le split de validation (pas le test — évite le data leakage)
seuil_optimal = plot_roc(model, val_dl, device)

## 10 — Courbe de loss (train + validation)

In [ ]:
# Reconstruction des listes pour le tracé
# val_loss_list contient des tuples (epoch, loss) car la validation
# n'est pas effectuée à chaque epoch
val_epochs = [x[0] for x in val_loss_list]
val_losses = [x[1] for x in val_loss_list]

plt.figure(figsize=(10, 5))
# Train loss : calculée à chaque epoch → courbe continue
plt.plot(range(1, len(loss_list) + 1), loss_list,
         marker='o', markersize=3, label='Train loss', color='steelblue')
# Val loss : calculée tous les 5 epochs → courbe éparse (marqueurs carrés)
plt.plot(val_epochs, val_losses,
         marker='s', markersize=5, linestyle='--', color='red',
         label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (FocalTversky)')
plt.title('Loss moyenne (BCE Focal + Tversky) — DeepLabV3+ V5')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
# Sauvegarde pour inclusion dans le rapport
plt.savefig(f'loss_curve_deeplabv3plus_v5_{num_epochs}.png', dpi=150)
plt.show()

## 11 — Évaluation finale sur les 3 splits

Le **meilleur checkpoint** (basé sur le F1 validation) est chargé pour l'évaluation finale.
On évalue sur les 3 splits avec le seuil optimal dérivé de la courbe PR.

La cohérence des métriques entre train/val/test confirme l'absence d'overfitting
et l'efficacité de la stratification IPS.

In [ ]:
# Chargement du meilleur checkpoint (critère : F1 validation max)
# map_location=device : compatibilité GPU/CPU (le checkpoint peut avoir été
# sauvegardé sur GPU et chargé sur CPU, ou vice versa)
model.load_state_dict(
    torch.load('best_model_deeplabv3plus_v5.pth', map_location=device))
model.eval()

print('=' * 70)
print('ÉVALUATION FINALE — DeepLabV3+ V5')
print('=' * 70)

# Évaluation sur les 3 splits avec le seuil optimal (issu de la courbe PR)
all_metrics = {}
for name, loader in [('TRAIN', train_dl), ('VALIDATION', val_dl),
                     ('TEST', test_dl)]:
    print(f'\n{name}')
    all_metrics[name] = evaluate_loader(
        model, loader, device, seuil=seuil_optimal)

# Tableau récapitulatif
print('\n' + '=' * 60)
print(f'Meilleur F1 Validation : {best_val_f1:.3f}')
print('Dataset      | IoU côte | mIoU  | Prec  | Rec   | F1    | Acc')
print('-' * 60)
for name, m in all_metrics.items():
    print(f'{name:12} | {m["IoU_côte"]:5.3f} | {m["mIoU"]:5.3f} | '
          f'{m["precision"]:5.3f} | {m["recall"]:5.3f} | '
          f'{m["f1"]:5.3f} | {m["accuracy"]:5.3f}')

## 12 — Inférence sur image complète avec diagnostic

### Pipeline sliding-window
Le modèle ne peut traiter que des tuiles 256×256 px. Pour prédire sur l'image Pléiades
complète (4695×2344 px), on utilise une fenêtre glissante avec stride=128 px (50% overlap).
Les probabilités des zones de recouvrement sont **moyennées**, ce qui lisse les artefacts
de jonction entre tuiles.

### Seuil d'inférence : 0.5 (pas seuil_optimal)
Le seuil optimal calculé sur tuiles de validation peut être trop élevé pour
la reconstruction sliding-window : les jonctions entre tuiles reçoivent des probabilités
plus faibles que l'intérieur des tuiles → fragments de coastline déconnectés.
SEUIL=0.5 produit une prédiction continue et géomorphologiquement interprétable.

### Filtrage par composantes connexes
Après binarisation, toutes les composantes connexes de moins de `MIN_SIZE=5000` pixels
sont supprimées. Ce filtre élimine les faux positifs isolés (ombres, bords de champs,
arêtes de bâtiments) dont la taille est bien inférieure à la ligne de côte principale
(~15000 pixels).

### Sortie
Le masque binaire final est sauvegardé en GeoTIFF géoréférencé (Lambert 93, EPSG:2154),
directement importable dans QGIS pour analyse géomorphologique.

In [ ]:
def diagnose_and_infer(model, input_path, device, tile_size=256, stride=128,
                       seuil=None, min_size=5000, out_dir=None):
    """Pipeline d'inférence complet avec diagnostic de la probability map.
    
    1. Découpe l'image en tuiles avec fenêtre glissante (stride=128)
    2. Prédit une probabilité par pixel pour chaque tuile
    3. Moyenne les probabilités des zones de recouvrement
    4. Diagnostique la distribution des probabilités (détecte le problème V4)
    5. Binarise avec un seuil fixe ou adaptatif (percentile 99.5)
    6. Filtre les petites composantes connexes (faux positifs)
    7. Sauvegarde masque binaire + probability map en GeoTIFF
    
    Paramètres
    ----------
    seuil : float or None
        Si None : seuil adaptatif = percentile 99.5 des probabilités.
        Garantit que ~0.5% des pixels sont détectés comme côte avant
        post-traitement, indépendamment du niveau absolu des probabilités.
        Si float : seuil fixe (0.5 recommandé pour V5).
    """
    import os
    infer_transform = get_val_transform()  # Normalisation ImageNet uniquement
    base_name = os.path.splitext(os.path.basename(input_path))[0]

    with rasterio.open(input_path) as src:
        W, H           = src.width, src.height
        crs            = src.crs if src.crs else CRS.from_string('EPSG:2154')
        transform_glob = src.transform  # Géoréférencement à conserver
        print(f'\n{base_name}  ({W}×{H}) — stride={stride}px')

        # Tableaux d'accumulation : somme des probabilités et comptage des tuiles
        # par pixel (pour le moyennage des zones de recouvrement)
        accum = np.zeros((H, W), dtype=np.float32)
        count = np.zeros((H, W), dtype=np.float32)

        # Génération des positions de la fenêtre glissante.
        # Le dernier terme (H-tile_size) gère le cas où H n'est pas multiple de stride
        # → garantit que les pixels du bord droit/bas sont bien couverts.
        tops  = list(range(0, H - tile_size + 1, stride)) + \
                ([H - tile_size] if H % stride != 0 else [])
        lefts = list(range(0, W - tile_size + 1, stride)) + \
                ([W - tile_size] if W % stride != 0 else [])
        print(f'  Nombre de tuiles : {len(tops) * len(lefts)}')

        for top in tqdm(tops, desc='Inférence'):
            for left in lefts:
                # Lecture de la tuile courante depuis le GeoTIFF (sans charger
                # toute l'image en mémoire grâce aux rasterio Windows)
                tile_data = src.read(
                    window=Window(left, top, tile_size, tile_size))
                # Conversion CHW → HWC et cast uint8 pour albumentations
                img_hwc   = np.moveaxis(
                    tile_data[:3].astype(np.uint8), 0, -1)
                # Application de la normalisation ImageNet
                augmented = infer_transform(image=img_hwc)
                # Ajout de la dimension batch (unsqueeze(0)) : [1, 3, 256, 256]
                tensor    = augmented['image'].unsqueeze(0).to(device)

                with torch.no_grad():
                    # sigmoid appliqué sur les logits → probabilités [0, 1]
                    # [0, 0] : supprime les dimensions batch et canal
                    prob = torch.sigmoid(
                        model(tensor))[0, 0].cpu().numpy()

                # Accumulation des probabilités pour moyennage ultérieur
                accum[top:top+tile_size, left:left+tile_size] += prob
                count[top:top+tile_size, left:left+tile_size] += 1.0

    # Moyennage : chaque pixel reçoit la moyenne des probabilités
    # de toutes les tuiles qui le couvrent (4 en zone centrale avec stride=128)
    mean_prob = accum / np.maximum(count, 1.0)  # np.maximum évite la division par zéro

    # ── Diagnostic automatique de la probability map ──────────────────────
    # Détecte le problème V4 (probas toutes < 0.1) et recommande un seuil
    print('\n── Diagnostic probability map ──')
    print(f'  min={mean_prob.min():.4f} | max={mean_prob.max():.4f} | '
          f'mean={mean_prob.mean():.4f}')
    p995 = np.percentile(mean_prob, 99.5)
    p999 = np.percentile(mean_prob, 99.9)
    print(f'  p99.5={p995:.4f} | p99.9={p999:.4f}')

    if mean_prob.max() < 0.1:
        print('  ⚠ ATTENTION : probabilités max < 0.1 → '
              'possible résidu du problème pos_weight élevé.\n'
              '  Vérifier que le modèle V5 est bien chargé (pas V4).')
    elif mean_prob.max() < 0.3:
        print('  ⚠ Probabilités faibles (max < 0.3) → seuil adaptatif recommandé.')
    else:
        print('  ✓ Distribution des probabilités normale (max > 0.3).')

    # Seuil adaptatif basé sur le percentile 99.5 si seuil non fourni
    if seuil is None:
        seuil_infer = float(p995)
        print(f'  Seuil adaptatif (p99.5) : {seuil_infer:.4f}')
    else:
        seuil_infer = seuil
        print(f'  Seuil fourni : {seuil_infer:.4f}')

    # ── Binarisation + filtrage par composantes connexes ─────────────────
    final_mask = (mean_prob > seuil_infer).astype(np.uint8)
    print(f'  Pixels détectés avant CC : {final_mask.sum():,}')

    # Étiquetage des composantes connexes (8-connectivité par défaut)
    labeled, num_features = ndimage.label(final_mask)
    # Taille de chaque composante
    component_sizes = ndimage.sum(
        final_mask, labeled, range(1, num_features + 1))
    # Ne conserve que les composantes >= min_size pixels
    # La ligne de côte principale contient ~15000 px ; les faux positifs < 5000 px
    cleaned_mask = np.zeros_like(final_mask)
    for i, size in enumerate(component_sizes, start=1):
        if size >= min_size:
            cleaned_mask[labeled == i] = 1
    final_mask = cleaned_mask
    print(f'  Pixels côte après CC (min_size={min_size}) : {final_mask.sum():,}')

    # ── Sauvegarde en GeoTIFF géoréférencé ───────────────────────────────
    if out_dir is None:
        out_dir = os.path.dirname(input_path)
    os.makedirs(out_dir, exist_ok=True)

    out_profile = {
        'driver': 'GTiff', 'dtype': 'uint8',
        'width': W, 'height': H, 'count': 1,
        'crs': crs,                # Système de coordonnées (Lambert 93, EPSG:2154)
        'transform': transform_glob,  # Géoréférencement pixel→coordonnées
    }

    suffix = f'_deeplabv3plus_v5_stride{stride}'
    mask_path = os.path.join(out_dir, f'{base_name}{suffix}.tif')
    prob_path = os.path.join(out_dir, f'{base_name}_probmap{suffix}.tif')

    # Sauvegarde du masque binaire (uint8 : 0 ou 1)
    with rasterio.open(mask_path, 'w', **out_profile) as dst:
        dst.write(final_mask[np.newaxis, :, :])  # np.newaxis ajoute la dim. de bande
    # Sauvegarde de la probability map (float32 : valeurs continues [0,1])
    # Utile pour le diagnostic et le choix du seuil
    with rasterio.open(prob_path, 'w',
                       **{**out_profile, 'dtype': 'float32'}) as dst:
        dst.write(mean_prob[np.newaxis, :, :])

    print(f'  Masque    : {mask_path}')
    print(f'  Prob map  : {prob_path}')
    return mean_prob, final_mask


print('Fonction diagnose_and_infer() prête.')

## 13 — Lancement de l'inférence

Traite tous les fichiers .tif du dossier `input_folder`.

**Paramètres à adapter :**
- `input_folder` : dossier contenant les images Pléiades à prédire
- `mosaic_output_root` : dossier de sortie pour les masques et probability maps
- `SEUIL` : 0.5 par défaut ; passer `None` pour le seuil adaptatif automatique

In [ ]:
# ── À ADAPTER  ──────────────────────────────────────
input_folder       = r'\Input_pred'#r'C:\Users\Hima\Desktop\DeeplabV3+\Input_pred'
mask_output_root   = r'\Masques_Georef_DLV3'#r'C:\Users\Hima\Desktop\DeeplabV3+\Masques_Georef_DLV3'
mosaic_output_root = r'\Masks_pred_DLV3'#r'C:\Users\Hima\Desktop\DeeplabV3+\Masks_pred_DLV3'
# ─────────────────────────────────────────────────────────────────────────────

tile_size = 256
STRIDE    = 128   # 50% de recouvrement → lisse les artefacts de jonction
MIN_SIZE  = 5000  # Taille minimale des composantes connexes à conserver

# SEUIL=0.5 recommandé pour V5 (les probabilités sont correctement calibrées).
# Passer SEUIL=None pour activer le seuil adaptatif (p99.5) si le masque est vide.
SEUIL = 0.5

# Chargement du meilleur checkpoint
model.load_state_dict(
    torch.load('best_model_deeplabv3plus_v5.pth', map_location=device))
model.to(device).eval()

# Traitement de tous les fichiers .tif du dossier d'entrée
for tif_file in [f for f in os.listdir(input_folder)
                 if f.lower().endswith('.tif')]:
    input_path = os.path.join(input_folder, tif_file)
    mean_prob, final_mask = diagnose_and_infer(
        model, input_path, device,
        tile_size=tile_size,
        stride=STRIDE,
        seuil=SEUIL,       # None = seuil adaptatif automatique (p99.5)
        min_size=MIN_SIZE,
        out_dir=mosaic_output_root)

print('\nInférence terminée.')

## 14 — Dilatation du masque pour visualisation

La ligne de côte prédite est une structure très fine (1-2 pixels de large),
difficile à visualiser sur l'image complète. On applique une **dilatation morphologique**
avec un élément structurant carré de rayon 5 pixels pour élargir la ligne et
la rendre visible à l'échelle de l'image entière.

**Important :** Ce masque dilaté est utilisé uniquement pour la visualisation.
Pour toute analyse géométrique (mesure de recul, comparaison temporelle),
utiliser le masque non dilaté.

In [ ]:
# ── Étape 1 : conversion du masque binaire (0/1) en uint8 (0/255) ─────────
# Certains logiciels SIG attendent des valeurs 0/255 pour les masques binaires
#with rasterio.open(r"C:\\Users\\Hima\\Desktop\\DeeplabV3+\\Masks_pred_DLV3\\decoupe_deeplabv3plus_v5_stride128.tif") as src:
with rasterio.open(r"\\Masks_pred_DLV3\\decoupe_deeplabv3plus_v5_stride128.tif") as src:
    m = src.read(1)              # Lecture de la 1ère bande
    m255 = (m * 255).astype("uint8")  # 0→0, 1→255
with rasterio.open(
    r"\\Masks_pred_DLV3\\decoupe_deeplabv3plus_v5_stride128_vis.tif",#r"C:\\Users\\Hima\\Desktop\\DeeplabV3+\\Masks_pred_DLV3\\decoupe_deeplabv3plus_v5_stride128_vis.tif",
    "w",
    driver="GTiff",
    height=m255.shape[0],
    width=m255.shape[1],
    count=1,
    dtype="uint8",
    crs=src.crs,
    transform=src.transform,
) as dst:
    dst.write(m255, 1)


# ── Étape 2 : dilatation morphologique pour la visualisation ──────────────
input_path  = r"\Masks_pred_DLV3\decoupe_deeplabv3plus_v5_stride128_vis.tif"#r"C:\Users\Hima\Desktop\DeeplabV3+\Masks_pred_DLV3\decoupe_deeplabv3plus_v5_stride128_vis.tif"
output_path = r"\Masks_pred_DLV3\decoupe_deeplabv3plus_v5_stride128_dilate.tif"#r"C:\Users\Hima\Desktop\DeeplabV3+\Masks_pred_DLV3\decoupe_deeplabv3plus_v5_stride128_dilate.tif"

with rasterio.open(input_path) as src:
    profile = src.profile  # Copie de tous les métadonnées (CRS, transform, etc.)
    mask = src.read(1).astype(bool)  # Conversion en booléen pour binary_dilation
    print(f"Avant dilatation: {mask.sum():,} pixels blancs")
    
    # Élément structurant carré de (2r+1)×(2r+1) pixels
    # r=5 → carré 11×11 : élargit la ligne de ~5 pixels de chaque côté
    DILATATION_RADIUS = 5
    struct = np.ones((2*DILATATION_RADIUS+1, 2*DILATATION_RADIUS+1))
    # binary_dilation : dilatation morphologique binaire
    # Chaque pixel True dans mask → le carré 11×11 centré sur lui devient True
    mask_dilated = binary_dilation(mask, structure=struct).astype(np.uint8)
    print(f"Après dilatation: {mask_dilated.sum():,} pixels blancs")
    
    # Conversion en uint8 pour sauvegarde (255=blanc, 0=noir pour visualisation)
    mask_dilated_uint8 = (mask_dilated * 255).astype(np.uint8)
    
    # Sauvegarde avec les mêmes métadonnées géographiques que l'original
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(mask_dilated_uint8, 1)
    
    print(f"Masque dilaté sauvé : {output_path}")